In [1]:
import os
os.chdir('/workspace/7e5300a5-9b47-4219-b422-9741df37baef')
print(os.listdir('.'))


['.prompts', '.config', '.kernel_llm_logs_1.txt', 'zeta_zeros_N5000_dps50.npy', '-PROMPT-v6-DATASET.md', 'zeta_delta_zeros_N5000_dps50.npy', 'memory', 'ldh_zeros_N5000_dps50.npy']


In [2]:
import numpy as np
zeta = np.load('zeta_zeros_N5000_dps50.npy', allow_pickle=False)
zeta_delta = np.load('zeta_delta_zeros_N5000_dps50.npy', allow_pickle=False)
ldh = np.load('ldh_zeros_N5000_dps50.npy', allow_pickle=False)
print(zeta.shape, zeta.dtype, zeta[:3])
print(zeta_delta.shape, zeta_delta.dtype, zeta_delta[:3], zeta_delta[1000])
print(ldh.shape, ldh.dtype, ldh[:3])


(5000,) <U51 ['14.134725141734693790457251983562470270784257115699'
 '21.022039638771554992628479593896902777334340524903'
 '25.010857580145688763213790992562821818659549672558']
(5000,) [('re', '<U10'), ('im', '<U51')] [('0.5', '14.134725141734693790457251983562470270784257115699')
 ('0.5', '21.022039638771554992628479593896902777334340524903')
 ('0.5', '25.010857580145688763213790992562821818659549672558')] ('0.6', '1420.4165263237511360343752509329151597418813931128')
(5000,) <U51 ['5.0941598445710949256987955170797974750670744531091'
 '8.9399144080965232447176128773475666256121616213856'
 '12.133545425800742403702547753765281383126636124036']


In [3]:

# Plan summary:
# - Compute Li coefficients λ_n for n ≤ 200 using formula:
# λ_n = Σ_ρ [1 - (1 - 1/ρ)^n]
# For real list of imaginary parts γ on critical line, the pair (1/2 ± iγ) contributes:
# 2 Re[1 - (1 - 1/(1/2+iγ))^n]
# - For zeta_delta, use actual real parts (mostly 0.5, 20 zeros 0.6) and pair with their conjugate (ρ, conj(ρ)) — since γ_n>0 only and we treat positive zeros, conj pair only.
# - For L_DH we use the same: each zero on the critical line ρ = 1/2 + iγ, plus conjugate; if L_DH has off-line zeros they aren't in this list (this is the critical line scan).
#
# Note: spec says L_DH has off-line zeros, but the zero list (ldh_zeros_N5000_dps50.npy) is described as "critical-line zeros" — these are zeros of the symmetric function on Re(s)=1/2. L_DH may also have off-line zeros not present in this list.
# This is an important caveat — Li sum over only critical-line zeros may be incomplete.
#
# Since L(χ) and L(Δ,s) zero lists are NOT in the workspace, we cannot compute λ_n for them.
# We must report this clearly.

# Two missing inputs: L(χ) and L(Δ,s) zero lists.
# Let's check if data-retrieval can get them quickly. Otherwise, we proceed with the 3 we have and clearly report the limitation.

# First, do exploratory check on memory and any cached lchi or delta files anywhere
import subprocess
result = subprocess.run(['find', '/workspace', '-name', '*.npy', '-type', 'f'], capture_output=True, text=True, timeout=30)
print(result.stdout)


/workspace/7e5300a5-9b47-4219-b422-9741df37baef/zeta_zeros_N5000_dps50.npy
/workspace/7e5300a5-9b47-4219-b422-9741df37baef/zeta_delta_zeros_N5000_dps50.npy
/workspace/7e5300a5-9b47-4219-b422-9741df37baef/ldh_zeros_N5000_dps50.npy



In [4]:

# We have only 3 of 5 zero lists. Per data integrity rules, we should not fabricate.
# Let's see if LMFDB has L(Δ, 1.12.a.a) zeros that can be quickly downloaded, and similarly L(χ_4 mod 5) via cypari2.
# First check if cypari2 is installed.
import importlib
for pkg in ['cypari2', 'mpmath', 'lmfdb']:
 try:
 m = importlib.import_module(pkg)
 print(pkg, 'OK', getattr(m, '__version__', '?'))
 except Exception as e:
 print(pkg, 'MISSING:', e)


cypari2 MISSING: No module named 'cypari2'
mpmath OK 1.3.0
lmfdb MISSING: No module named 'lmfdb'


In [5]:

# Try fetching from LMFDB directly - the modular form 1.12.a.a zeros
import urllib.request
import json

# LMFDB has an API. Let's try a few known endpoints.
urls_to_try = [
 "https://www.lmfdb.org/api/lfunctions/zeros/?Lhash=1.12.a.a&_format=json",
 "https://www.lmfdb.org/L/ModularForm/GL2/Q/holomorphic/1/12/a/a/",
 "https://www.lmfdb.org/api/lfunctions/Lfunctions/?label=1.12.a.a&_format=json",
]
for url in urls_to_try:
 try:
 req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
 with urllib.request.urlopen(req, timeout=20) as r:
 data = r.read().decode('utf-8', errors='ignore')
 print(url, '->', data[:300])
 except Exception as e:
 print(url, 'ERR', e)


https://www.lmfdb.org/api/lfunctions/zeros/?Lhash=1.12.a.a&_format=json ERR HTTP Error 404: Not Found


https://www.lmfdb.org/L/ModularForm/GL2/Q/holomorphic/1/12/a/a/ -> <!DOCTYPE html>
<html lang="en">
 <head>
 <meta http-equiv="Content-Type" content="text/html; charset=UTF-8">
 <meta http-equiv="X-UA-Compatible" content="IE=edge,chrome=1">
 <meta name="description" content="Welcome to the LMFDB, the database of L-functions, modular forms, and related obj
https://www.lmfdb.org/api/lfunctions/Lfunctions/?label=1.12.a.a&_format=json ERR HTTP Error 404: Not Found


In [6]:

# Per data integrity, generating 5000 high-precision zeros for L(χ_4 mod 5) and L(Δ,s) 
# in this session via mpmath would take many hours (way beyond budget).
# The L(χ) and L(Δ,s) zero lists are documented as MISSING from the workspace.
# We must report this limitation.
#
# Decision: Proceed with what we have (ζ, ζ_δ, L_DH) and compute Li coefficients rigorously.
# For the two missing controls, we report null/cannot complete.
# We can also attempt a modest mpmath zero generation just to enable a small-N
# sanity check on L(χ_4 mod 5) — but only if computational budget allows. 
# Actually, mpmath.lerchphi and similar are slow; let's first see what we can do with mpmath.

import mpmath as mp
mp.mp.dps = 50

# Test: compute the first few zeros of L(s, chi_4 mod 5) using mpmath
# Note: mpmath has dirichlet character support. Let's check chi_4 mod 5.
# The character chi_4 mod 5: order 4, this is the unique nontrivial character of order 4 mod 5.
# mpmath.dirichlet(s, [chi_values]) computes the Dirichlet L-function with given character values.

# The character chi_4 mod 5: chi(1)=1, chi(2)=i, chi(3)=-i, chi(4)=-1, chi(5)=0
# Period 5 character with values
chi_vals = [0, 1, 1j, -1j, -1, 0] # mpmath uses chi[n%period], with chi[0] for n=0
# Actually mpmath.dirichlet has its own conventions

# Test the L-function at s=1/2 + 14i (just check it works)
def L_chi(s):
 return mp.dirichlet(s, chi_vals[1:5]) # mpmath wants character values chi[1..p-1]?

# Actually let's read mpmath docs
help(mp.dirichlet)


Help on method dirichlet in module mpmath.functions.zeta:

dirichlet(s, chi=[1], derivative=0) method of mpmath.ctx_mp.MPContext instance
 Evaluates the Dirichlet L-function

 .. math ::

 L(s,\chi) = \sum_{k=1}^\infty \frac{\chi(k)}{k^s}.

 where `\chi` is a periodic sequence of length `q` which should be supplied
 in the form of a list `[\chi(0), \chi(1), \ldots, \chi(q-1)]`.
 Strictly, `\chi` should be a Dirichlet character, but any periodic
 sequence will work.

 For example, ``dirichlet(s, [1])`` gives the ordinary
 Riemann zeta function and ``dirichlet(s, [-1,1])`` gives
 the alternating zeta function (Dirichlet eta function).

 Also the derivative with respect to `s` (currently only a first
 derivative) can be evaluated.

 **Examples**

 The ordinary Riemann zeta function::

 >>> from mpmath import *
 >>> mp.dps = 25; mp.pretty = True
 >>> dirichlet(3, [1]); zeta(3)
 1.202056903159594285399738
 1.202056903159594285399738
 >>> dirichlet(1, [1])
 +inf

 The alternating zeta functi

In [7]:

# Generating 5000 zeros for L(chi_4 mod 5) using mpmath would be very slow.
# Approach: We'll focus on the core hypothesis test using available data (ζ, ζ_δ, L_DH).
# The hypothesis prediction is asymmetric: L_DH should produce λ_n < 0, while other 3 should stay positive.
# Showing this signature on L_DH is the primary test. ζ serves as positive control.
# Without L(χ) and L(Δ,s) lists we cannot verify they remain λ_n>0, but ζ and ζ_δ (which has imag=real ζ imag parts plus 20 perturbed real parts) provide GRH-class comparators.
# 
# Also: ζ_δ's imaginary parts are identical to ζ. Since the Li formula DOES use 1/ρ = 1/(σ+iγ), it IS sensitive to real-part perturbations.
# So ζ_δ provides a controlled comparator where 20 zeros at index 1000-1019 have Re(ρ)=0.6 instead of 0.5.

# Let's now build the Li coefficient computation.
import mpmath as mp
import numpy as np

def load_zeros_str_list(arr, structured=False):
 """Returns list of mpmath complex zeros (ρ = re + i*im), one per entry."""
 out = []
 if structured:
 for entry in arr:
 re = mp.mpf(str(entry['re']))
 im = mp.mpf(str(entry['im']))
 out.append(mp.mpc(re, im))
 else:
 for s in arr:
 im = mp.mpf(str(s))
 out.append(mp.mpc(mp.mpf('0.5'), im))
 return out

# Convert all three
mp.mp.dps = 50
zeta_rho = load_zeros_str_list(zeta, structured=False)
zeta_delta_rho = load_zeros_str_list(zeta_delta, structured=True)
ldh_rho = load_zeros_str_list(ldh, structured=False)
print(len(zeta_rho), zeta_rho[0], zeta_rho[-1])
print(len(zeta_delta_rho), zeta_delta_rho[1000], zeta_delta_rho[999])
print(len(ldh_rho), ldh_rho[0])


5000 (0.5 + 14.134725141734693790457251983562470270784257115699j) (0.5 + 5447.8619983012998564121586734642921676829672006076j)
5000 (0.6 + 1420.4165263237511360343752509329151597418813931128j) (0.5 + 1419.4224809459956864659890380799168192321006010642j)
5000 (0.5 + 5.0941598445710949256987955170797974750670744531091j)


In [8]:

# Efficient Li-coefficient computation.
# For each ρ = σ + iγ on the upper half plane, the conjugate ρ̄ = σ - iγ is also a zero.
# So pair contribution = [1 - (1 - 1/ρ)^n] + [1 - (1 - 1/ρ̄)^n] = 2 * Re[1 - (1 - 1/ρ)^n]
# = 2 * (1 - Re[(1 - 1/ρ)^n])
#
# For very large n (up to 200), w = 1 - 1/ρ has |w| < 1 only if Re(1/ρ) > 0 i.e. σ > 0 always; 
# but we need |w|<1: |1 - 1/ρ|^2 = 1 - 2σ/|ρ|^2 + 1/|ρ|^2 < 1 iff 1 - 2σ < 0 iff σ > 1/2. 
# For σ = 1/2: |w|^2 = 1 - 1/|ρ|^2 + 1/|ρ|^2 = ... let's compute:
# |1 - 1/ρ|^2 = (1 - σ/|ρ|^2)^2 + (γ/|ρ|^2)^2 = 1 - 2σ/|ρ|^2 + (σ^2+γ^2)/|ρ|^4 = 1 - 2σ/|ρ|^2 + 1/|ρ|^2
# So |w|^2 = 1 + (1 - 2σ)/|ρ|^2. 
# For σ=1/2: |w|=1 exactly. For σ=0.6: |w|<1 (off-critical inside). For σ>0.5 zero off critical line on right, |w|<1.
# So for GRH-satisfying zeros on σ=1/2, |w|=1 → no growth/decay of w^n; just oscillation.
# For L_DH off-line zeros (σ < 0.5), |w|>1 → w^n grows exponentially → Re[(1-1/ρ)^n] can be large negative, making λ_n term large positive (since 1 - Re[w^n]).
# But the list ldh_rho only contains CRITICAL LINE zeros (σ=0.5), not off-line zeros.
# Hmm.
#
# Wait — the data description says ldh_zeros is "critical-line zeros of L_DH". L_DH has zeros BOTH on and off the critical line. The off-line zeros are responsible for GRH failure.
# But the file only contains critical-line zeros, NOT off-line zeros.
# 
# So the Li computation here uses ONLY critical-line zeros of L_DH. This is incomplete:
# The Li-coefficient λ_n = Σ over ALL zeros — to detect negative λ_n we need the off-line zeros included.
# 
# The off-line zeros of L_DH per spec: 4 known off-line points (0.808517,85.699348), (0.650786,114.163343), (0.574355,166.479306), (0.724258,176.702461), and likely many more.
# 
# This is critical: with only critical-line zeros, L_DH's λ_n is treated identically to ζ's (in structure), and we may NOT see negative λ_n.
# 
# Let me first compute what we have, and ALSO compute with the off-line zeros added (4 known), to see if that's enough to flip sign.

# Build the standard symmetric Li coefficient computation:
def compute_li_coefficients(rho_list, n_max=200, dps=50):
 """Compute λ_n = Σ_ρ [1 - (1-1/ρ)^n], where ρ runs over zeros AND their conjugates.
 
 Input: rho_list = list of zeros in the UPPER half plane (γ > 0).
 Each pair (ρ, ρ̄) contributes 2·Re(1 - (1-1/ρ)^n).
 """
 mp.mp.dps = dps
 one = mp.mpf(1)
 # For each ρ, compute z = 1 - 1/ρ, then powers up to n_max
 # We accumulate λ_n incrementally.
 lambdas = [mp.mpf(0)] * (n_max + 1) # lambdas[n] for n in 0..n_max
 for rho in rho_list:
 z = one - one / rho # complex
 # cumulative powers: z, z^2, ..., z^n_max
 zp = mp.mpc(1, 0) # z^0
 for n in range(1, n_max + 1):
 zp = zp * z
 # contribution from (ρ, conj(ρ)) = 2 * Re(1 - z^n) = 2*(1 - Re(z^n))
 lambdas[n] += 2 * (one - zp.real)
 return lambdas[1:] # return for n = 1..n_max

# Estimate cost: 5000 zeros × 200 powers × complex mult = 1M ops at dps=50.
# Each mpmath complex multiply ~ tens of microseconds; total ~ tens of seconds. Should be OK.

import time
t0 = time.time()
# Test with small subset
test_lam = compute_li_coefficients(zeta_rho[:200], n_max=50, dps=50)
print('time for 200 zeros, n_max=50:', time.time()-t0, 'seconds')
print('λ_1 to λ_5:', [float(x) for x in test_lam[:5]])


time for 200 zeros, n_max=50: 0.07926464080810547 seconds
λ_1 to λ_5: [0.02103486031742673, 0.08410234441738831, 0.18909130542077532, 0.3358169767953985, 0.5240216874492751]


In [9]:

# Cost estimate at full scale
# 5000 × 200 / (200 × 50) = 100x more work → ~8 seconds. Great.
# Compute full Li coefficients for the three lists, n up to 200, at dps=50.

import time
n_max = 200
t0 = time.time()
lam_zeta_5000 = compute_li_coefficients(zeta_rho, n_max=n_max, dps=50)
print('zeta:', time.time()-t0, 's')

t0 = time.time()
lam_zetad_5000 = compute_li_coefficients(zeta_delta_rho, n_max=n_max, dps=50)
print('zeta_delta:', time.time()-t0, 's')

t0 = time.time()
lam_ldh_5000 = compute_li_coefficients(ldh_rho, n_max=n_max, dps=50)
print('ldh:', time.time()-t0, 's')


zeta: 7.098062515258789 s


zeta_delta: 7.138482570648193 s


ldh: 7.194303750991821 s


In [10]:

# Examine the Li coefficient sequences. Note: λ_n for truncated zero list will NOT match
# the true λ_n values. The "true" Li criterion uses ALL zeros (infinite sum). Truncation introduces error.
# But under GRH, the partial sums should be positive and grow. Negative values signal off-line zeros OR insufficient truncation.

import numpy as np
lz = np.array([float(x) for x in lam_zeta_5000])
lzd = np.array([float(x) for x in lam_zetad_5000])
ld = np.array([float(x) for x in lam_ldh_5000])

print('--- ζ (5000 zeros) ---')
print('λ_1..λ_5:', lz[:5])
print('λ_100, λ_150, λ_200:', lz[99], lz[149], lz[199])
print('Min λ_n for 1≤n≤200:', lz.min(), 'at n =', lz.argmin()+1)
print('Negative count:', (lz<0).sum())
print()
print('--- ζ_δ (5000 zeros, 20 perturbed) ---')
print('λ_1..λ_5:', lzd[:5])
print('Min λ_n:', lzd.min(), 'at n =', lzd.argmin()+1)
print('Negative count:', (lzd<0).sum())
print()
print('--- L_DH (5000 critical-line zeros) ---')
print('λ_1..λ_5:', ld[:5])
print('Min λ_n:', ld.min(), 'at n =', ld.argmin()+1)
print('Negative count:', (ld<0).sum())
print('First few negative n:', np.where(ld<0)[0][:10]+1 if (ld<0).any() else 'none')


--- ζ (5000 zeros) ---
λ_1..λ_5: [0.02286887 0.09143839 0.2055974 0.36516111 0.56987183]
λ_100, λ_150, λ_200: 116.33544062876071 202.57367507347223 297.5826587055243
Min λ_n for 1≤n≤200: 0.022868873551074854 at n = 1
Negative count: 0

--- ζ_δ (5000 zeros, 20 perturbed) ---
λ_1..λ_5: [0.02287083 0.0914423 0.20560326 0.36516892 0.56988159]
Min λ_n: 0.022870825208673538 at n = 1
Negative count: 0

--- L_DH (5000 critical-line zeros) ---
λ_1..λ_5: [0.09630773 0.38351189 0.85651348 1.50700552 2.32375258]
Min λ_n: 0.09630772584745968 at n = 1
Negative count: 0
First few negative n: none


In [11]:

# IMPORTANT: With only critical-line zeros, L_DH shows λ_n > 0 for all n ≤ 200 (no negative coefficients).
# This is because λ_n's negativity in Li criterion REQUIRES off-line zeros.
# The ldh_zeros file only contains critical-line zeros.
# 
# Let's also verify by computing the "true" λ_n with the 4 known off-line zeros (and their reflections under functional eq)
# L_DH functional equation: zeros come in (ρ, 1-ρ, conj(ρ), 1-conj(ρ)) quadruples for off-line zeros.
# The 4 known off-line zeros from the spec:
off_line_4 = [
 (0.808517, 85.699348),
 (0.650786, 114.163343), # known transcription artifact
 (0.574355, 166.479306),
 (0.724258, 176.702461),
]
# For each off-line zero (σ, γ) with σ > 1/2, the symmetric partners are:
# ρ_1 = σ + iγ
# ρ_2 = (1-σ) + iγ
# ρ_3 = σ - iγ (conjugate)
# ρ_4 = (1-σ) - iγ
# In the Li sum, we include all 4. But when we use pair contribution as 2 Re[1-(1-1/ρ)^n], the pair is (ρ, conj(ρ)).
# So for each off-line zero we have TWO conjugate pairs: (ρ_1, ρ_3) and (ρ_2, ρ_4).

off_line_rhos_upper = []
mp.mp.dps = 50
for sig, gam in off_line_4:
 s_mp = mp.mpf(str(sig))
 g_mp = mp.mpf(str(gam))
 off_line_rhos_upper.append(mp.mpc(s_mp, g_mp)) # ρ
 off_line_rhos_upper.append(mp.mpc(1 - s_mp, g_mp)) # 1-conj(ρ) = (1-σ) + iγ

# Recompute Li using ldh_rho (critical-line) PLUS these off-line zeros (each with their conjugate)
ldh_rho_with_offline = ldh_rho + off_line_rhos_upper
t0 = time.time()
lam_ldh_with_offline = compute_li_coefficients(ldh_rho_with_offline, n_max=200, dps=50)
print('time:', time.time()-t0)

ld_off = np.array([float(x) for x in lam_ldh_with_offline])
print('L_DH + 4 off-line zeros:')
print('Min λ_n:', ld_off.min(), 'at n =', ld_off.argmin()+1)
print('Negative count:', (ld_off<0).sum())
print('First negative n:', np.where(ld_off<0)[0][:5]+1)
print('λ_1..λ_5:', ld_off[:5])
print('λ_100, λ_150, λ_200:', ld_off[99], ld_off[149], ld_off[199])


time: 7.110806465148926
L_DH + 4 off-line zeros:
Min λ_n: 0.09686968724070003 at n = 1
Negative count: 0
First negative n: []
λ_1..λ_5: [0.09686969 0.38575968 0.86157081 1.51599583 2.33779894]
λ_100, λ_150, λ_200: 188.20130843612597 310.8779827961299 437.4665952137902


In [12]:

# Even with the 4 known off-line zeros added, no negativity within n ≤ 200.
# That's because the contribution from off-line zeros: |1-1/ρ| > 1 means Re((1-1/ρ)^n) grows like |1-1/ρ|^n.
# For ρ = 0.808517 + 85.699i: |1-1/ρ|^2 = 1 + (1 - 2*0.808517)/|ρ|^2 = 1 - 0.617/7345 ≈ very close to 1.
# So growth is extremely slow. The off-line zeros are very far from real axis, so 1/ρ is tiny.
# To see negativity we'd need a LOT more off-line zeros (or zeros with small |γ|).

# Compute the |1-1/ρ| for the off-line zeros:
for sig, gam in off_line_4:
 rho = complex(sig, gam)
 w = 1 - 1/rho
 print(f"ρ=({sig},{gam}): |1-1/ρ|={abs(w):.8f}, growth^200 = {abs(w)**200:.4f}")


ρ=(0.808517,85.699348): |1-1/ρ|=0.99995800, growth^200 = 0.9916
ρ=(0.650786,114.163343): |1-1/ρ|=0.99998843, growth^200 = 0.9977
ρ=(0.574355,166.479306): |1-1/ρ|=0.99999732, growth^200 = 0.9995
ρ=(0.724258,176.702461): |1-1/ρ|=0.99999282, growth^200 = 0.9986


In [13]:

# Now let's compute the convergence diagnostics: λ_n with N=2000 vs N=5000 for ζ.
mp.mp.dps = 50
t0 = time.time()
lam_zeta_2000 = compute_li_coefficients(zeta_rho[:2000], n_max=200, dps=50)
print('zeta 2k:', time.time()-t0)

lz2k = np.array([float(x) for x in lam_zeta_2000])
delta_N = np.abs(lz - lz2k) # |λ_n(5000) - λ_n(2000)|
print('Δ_N at n=10,50,100,150,200:', delta_N[9], delta_N[49], delta_N[99], delta_N[149], delta_N[199])
print('Relative Δ_N/|λ_n| at n=200:', delta_N[199]/abs(lz[199]))
print('max Δ_N for n≤50:', delta_N[:50].max(), 'at n=', delta_N[:50].argmax()+1)
print('max Δ_N for n≤100:', delta_N[:100].max(), 'at n=', delta_N[:100].argmax()+1)
print('max Δ_N for n≤200:', delta_N[:200].max(), 'at n=', delta_N[:200].argmax()+1)


zeta 2k: 2.850008726119995
Δ_N at n=10,50,100,150,200: 0.02155025367871044 0.5387469699955219 2.1548707342666376 4.848019885495262 8.617608844236884
Relative Δ_N/|λ_n| at n=200: 0.02895870640353583
max Δ_N for n≤50: 0.5387469699955219 at n= 50
max Δ_N for n≤100: 2.1548707342666376 at n= 100
max Δ_N for n≤200: 8.617608844236884 at n= 200


In [14]:

# Δ_N grows linearly with n (since each added zero contributes ~ n/(2|ρ|^2) per spec)
# Truncation convergence: relative error at n=200 is ~3%; at n=50 it's ~0.5/26 ≈ 2%; reasonable.
# The expected asymptotic per Bombieri-Lagarias: λ_n ≈ (n/2) log n.
# 
# Now compute Q_N(φ) Weil quadratic form for N=2000 zeros of ζ using two kernels: bump (Gaussian) and Fejér.
# 
# The Weil explicit formula relates a sum over zeros to a sum over primes.
# The standard Weil quadratic form Q(φ):
# Q(φ) = Σ_ρ |φ̂(γ_ρ)|^2 ≥ 0 iff Weil positivity holds
# Specifically, given a test function h(r) and its Fourier transform g(u) = ∫ h(r) e^{iur} dr:
# Σ_γ h(γ) = h(i/2) + h(-i/2) - g(0) log π + (1/2π) ∫ h(r) Re[ψ(1/4+ir/2)] dr - 2 Σ_p Σ_k (log p / p^{k/2}) g(k log p)
# 
# For Weil positivity: choose h(r) = |H(r)|^2 ≥ 0 (Hermitian function), then RHS must be ≥ 0.
# 
# A cleaner formulation: given test functions φ_1, ..., φ_M (with compact support of φ̂), 
# define matrix Q_{ij} = <φ_i, φ_j>_Weil where the Weil pairing equals 
# Σ_ρ φ̂_i(γ)*conj(φ̂_j(γ))
# This is naturally PSD if we just sum over the zeros (it's a Gram matrix in L^2 of zero spectrum).
# The Weil explicit formula equates this to an arithmetic side: prime sum + archimedean.
# 
# Given the spec doesn't define Q_N precisely, we'll use a standard formulation:
# Use a basis of test functions {φ_k} on a fixed interval [0,T], compute the Gram matrix
# Q_{ij} = Σ_n φ_i(γ_n) φ_j(γ_n) (which is real and PSD by construction).
# 
# Actually that's trivially PSD. To make it a meaningful Weil-positivity test, one uses:
# Q_{ij} = Σ_n φ̂_i(γ_n) conj(φ̂_j(γ_n)) - (arithmetic side equivalent)
# But the standard Weil POSITIVITY test on critical-line zeros: 
# given φ_i = exp(-α_i (t-c_i)^2) bumps in frequency, compute the matrix 
# M_{ij} = Σ_{γ} φ_i(γ) φ_j(γ)
# Eigenvalues should be non-negative (trivially true). 
# 
# A common "Weil form" used in numerical experiments: 
# Q(φ) = h(i/2) + h(-i/2) - g(0)log(π) + arch - 2 Σ_p Σ_k (log p / p^{k/2}) g(k log p)
# RH ⟺ Q(φ) = Σ_ρ h(γ_ρ) ≥ 0 for h of the form |H|^2.
# 
# This is more involved. For this analysis, given the spec doesn't pin down a formula,
# we'll implement the cleanest interpretation:
# 
# Define a family of test functions parameterized by frequency centers c_k:
# φ_k(r) = bump_window(r) * cos(c_k r) etc.
# 
# Actually the most natural way: Let's use the explicit formula form and compute 
# Q(φ) = 2 Re Σ_n h(γ_n) − [arithmetic side]
# where h is chosen, and check the sign. Since we're told to compute Q_N(φ) for both bump and Fejér 
# kernels and find min eigenvalue, we'll compute a matrix Q indexed by a parametric family.
# 
# Given the ambiguity, I'll proceed pragmatically and clearly document the choice.

# Simplest concrete implementation:
# For a family of test functions {h_k(r) = φ((r-c_k)/σ)} k=1..M centered at c_k in [γ_1, γ_N], 
# build M×M matrix:
# Q_{ij} = Σ_n h_i(γ_n) h_j(γ_n)
# This is the discrete "Weil form" / Gram matrix on the zero spectrum.
# Compute λ_min(Q) for both:
# (a) Gaussian bump: h(r) = exp(-r^2/(2σ^2))
# (b) Fejér kernel: h(r) = (sin(r/2) / (r/2))^2 (Fourier transform of triangle)
# 
# This is what the spec seems to allude to. Let's go with this.

# First convert γ values to floats for N=2000 (no high precision needed for matrix here)
gamma_2000 = np.array([float(mp.mpf(str(z))) for z in zeta[:2000]])
print('γ range:', gamma_2000.min(), '-', gamma_2000.max())
print('N=2000, max γ ≈', gamma_2000[-1])


γ range: 14.134725141734695 - 2515.286482924713
N=2000, max γ ≈ 2515.286482924713


In [15]:

# Build Q_N matrix for both kernels.
# We'll use M=50 test functions centered at evenly spaced c_k in [γ_1, γ_N], width σ chosen to be ~ spacing.

import numpy as np
N = 2000
gammas = gamma_2000
M = 50 # number of test functions
centers = np.linspace(gammas[0]+5, gammas[-1]-5, M)
sigma = (gammas[-1] - gammas[0]) / (2*M) # bandwidth ~ spacing between centers
print('sigma:', sigma)

def gaussian_bump(r, c, s):
 return np.exp(-((r-c)/s)**2 / 2)

def fejer(r, c, s):
 x = (r - c) / s
 # sinc squared form
 out = np.where(np.abs(x) < 1e-12, 1.0, (np.sin(x/2) / (x/2))**2)
 return out

# Build Q matrices
def build_Q(kernel_fn):
 H = np.zeros((M, N))
 for i in range(M):
 H[i, :] = kernel_fn(gammas, centers[i], sigma)
 Q = H @ H.T # M × M Gram matrix
 return Q

Q_bump = build_Q(gaussian_bump)
Q_fejer = build_Q(fejer)

eig_bump = np.linalg.eigvalsh(Q_bump)
eig_fejer = np.linalg.eigvalsh(Q_fejer)

print('Q_bump shape:', Q_bump.shape)
print('Q_bump eigenvalues range:', eig_bump.min(), '..', eig_bump.max())
print('Q_bump min eigenvalue:', eig_bump.min())
print('Q_fejer eigenvalues range:', eig_fejer.min(), '..', eig_fejer.max())
print('Q_fejer min eigenvalue:', eig_fejer.min())


sigma: 25.011517577829782
Q_bump shape: (50, 50)
Q_bump eigenvalues range: 4.51618826833442 .. 71.32648586624848
Q_bump min eigenvalue: 4.51618826833442
Q_fejer eigenvalues range: -8.753200969739905e-14 .. 407.4174378746256
Q_fejer min eigenvalue: -8.753200969739905e-14


In [16]:

# Q_bump and Q_fejer are PSD by construction (Gram matrices), 
# the Q_fejer ~ -1e-13 is numerical noise consistent with zero (one eigenvalue is essentially 0).
# Q_bump has min eigenvalue ≈ 4.52, strictly positive.
# 
# Let me also verify by exploring with more test functions and via a different formulation closer to Weil's:
# The standard Weil positivity is meaningful when the kernel is not just any test function but specifically
# h(r) = |H(r)|^2 ≥ 0 — choose a family of |H_i|^2 and compute Σ_ρ h_i(γ_ρ).
# 
# In the standard formulation, the Weil quadratic form acts on H (not h directly):
# Q[H, K] = Σ_ρ H(γ_ρ) conj(K(γ_ρ)) - [arithmetic + archimedean terms]
# The arithmetic side keeps it from being trivially PSD.
# 
# Since the spec doesn't pin down the formula and the task is exploratory ("compute Q_N(φ) and its min eigenvalue"),
# we proceed with the Gram-matrix formulation, which is the cleanest "compute" interpretation,
# but acknowledge in the report that this trivially PSD form doesn't test Weil positivity per se.
# 
# Now perform convergence diagnostics S(n) at dps=80 vs dps=50. This is expensive but only for ζ at fewer zeros.

# To control cost, recompute zeta Li at dps=80 with N=2000 zeros, n_max=200.
mp.mp.dps = 80
zeta_rho_80 = []
for s in zeta[:2000]:
 im = mp.mpf(str(s))
 zeta_rho_80.append(mp.mpc(mp.mpf('0.5'), im))

import time
t0 = time.time()
lam_zeta_2000_dps80 = compute_li_coefficients(zeta_rho_80, n_max=200, dps=80)
print('time dps=80 2000 zeros:', time.time()-t0)


time dps=80 2000 zeros: 3.1322708129882812


In [17]:

# Compute precision-convergence S(n) = |λ_n(dps=80) - λ_n(dps=50)| at N=2000
lz2k_80 = np.array([float(x) for x in lam_zeta_2000_dps80])
S_n = np.abs(lz2k_80 - lz2k)
print('Precision convergence S(n) at N=2000:')
print('S at n=10,50,100,150,200:', S_n[9], S_n[49], S_n[99], S_n[149], S_n[199])
print('max S(n):', S_n.max(), 'at n=', S_n.argmax()+1)
# So precision is very stable; truncation Δ_N dominates.


Precision convergence S(n) at N=2000:
S at n=10,50,100,150,200: 0.0 0.0 0.0 0.0 0.0
max S(n): 0.0 at n= 1


In [18]:

# Precision is essentially perfect (dps=50 already saturates float64 representation we're using).
# Truncation error Δ_N dominates and grows linearly with n.
# 
# Save the Li coefficient arrays for archival.
np.savez('li_coefficients_N5000_dps50.npz',
 lam_zeta_N5000=lz,
 lam_zeta_delta_N5000=lzd,
 lam_ldh_N5000=ld,
 lam_ldh_with_4offline=ld_off,
 lam_zeta_N2000=lz2k,
 delta_N_zeta=delta_N,
 n_range=np.arange(1, 201))
print('saved')

# Now produce the final figure.
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(10, 10))

# Panel A: λ_n for all available functions across n=1..200
ax = axes[0]
n_arr = np.arange(1, 201)
ax.plot(n_arr, lz, label='ζ (N=5000)', color='C0', lw=1.6)
ax.plot(n_arr, lzd, label='ζ_δ (N=5000, 20 zeros perturbed)', color='C1', lw=1.2, linestyle='--')
ax.plot(n_arr, ld, label='L_DH critical-line zeros (N=5000)', color='C3', lw=1.6)
ax.plot(n_arr, ld_off, label='L_DH + 4 known off-line zeros', color='C2', lw=1.2, linestyle=':')
ax.axhline(0, color='k', lw=0.5, linestyle='-')
ax.set_xlabel('n')
ax.set_ylabel(r'$\lambda_n$')
ax.set_title(r'A. Li coefficients $\lambda_n$ for $n \leq 200$')
ax.legend(loc='upper left', fontsize=9)
ax.grid(alpha=0.3)

# Panel B: zoomed view on small λ_n to inspect possible negativity, plus L_DH and ζ truncation convergence
ax = axes[1]
ax.plot(n_arr, lz - lz2k, label=r'$\zeta$ truncation $\Delta_N(n) = \lambda_n(5000) - \lambda_n(2000)$', color='C0')
ax.plot(n_arr, np.zeros_like(n_arr), 'k-', lw=0.5)
ax.set_xlabel('n')
ax.set_ylabel(r'$\lambda_n$ difference')
ax.set_title(r'B. ζ truncation convergence: $\lambda_n(N=5000) - \lambda_n(N=2000)$')
ax.grid(alpha=0.3)
ax.legend(fontsize=9)

# Add summary text
fig.suptitle(r'Front II: Li coefficients across available controls (N=5000, dps=50)', fontsize=12)
plt.tight_layout()
plt.savefig('front_ii_li_coefficients.png', dpi=200, bbox_inches='tight')
plt.close()
print('Figure saved as front_ii_li_coefficients.png')


saved


Figure saved as front_ii_li_coefficients.png


In [19]:

# Summary table
print("=" * 80)
print("FRONT II RESULTS — Li coefficients λ_n, n=1..200, N=5000, dps=50")
print("=" * 80)
print(f"{'Function':<35} {'Min λ_n':>14} {'argmin':>8} {'#(λ_n<0)':>10}")
print('-'*80)
for label, arr in [('ζ', lz), ('ζ_δ (20 zeros perturbed)', lzd),
 ('L_DH (critical-line only)', ld),
 ('L_DH + 4 known off-line', ld_off)]:
 print(f"{label:<35} {arr.min():>14.6f} {arr.argmin()+1:>8d} {(arr<0).sum():>10d}")

print()
print("L(χ_4 mod 5) and L(Δ,s) zero lists: NOT AVAILABLE in workspace -> NOT COMPUTED.")
print()
print("ζ truncation convergence diagnostics (λ_n(5000) vs λ_n(2000)):")
for n in [1, 10, 50, 100, 150, 200]:
 print(f" n={n:>3}: λ_n(5000)={lz[n-1]:>12.6f}, λ_n(2000)={lz2k[n-1]:>12.6f}, Δ_N={delta_N[n-1]:>10.4f}, rel={delta_N[n-1]/abs(lz[n-1]):.4%}")

print()
print("ζ precision-convergence diagnostics S(n) at N=2000:")
print(f" max S(n) over n=1..200: {S_n.max():.2e}")
print()
print("Weil quadratic form Q_N at N=2000 (M=50 test functions):")
print(f" Gaussian bump kernel : λ_min(Q) = {eig_bump.min():.6f}")
print(f" Fejér kernel : λ_min(Q) = {eig_fejer.min():.2e} (numerically zero)")
print()
print("NOTE: Q_N here is the Gram matrix Σ_n h_i(γ_n) h_j(γ_n) which is PSD by construction;")
print("it does NOT test the arithmetic-side Weil positivity. Result documents this caveat.")


FRONT II RESULTS — Li coefficients λ_n, n=1..200, N=5000, dps=50
Function Min λ_n argmin #(λ_n<0)
--------------------------------------------------------------------------------
ζ 0.022869 1 0
ζ_δ (20 zeros perturbed) 0.022871 1 0
L_DH (critical-line only) 0.096308 1 0
L_DH + 4 known off-line 0.096870 1 0

L(χ_4 mod 5) and L(Δ,s) zero lists: NOT AVAILABLE in workspace -> NOT COMPUTED.

ζ truncation convergence diagnostics (λ_n(5000) vs λ_n(2000)):
 n= 1: λ_n(5000)= 0.022869, λ_n(2000)= 0.022653, Δ_N= 0.0002, rel=0.9423%
 n= 10: λ_n(5000)= 2.256656, λ_n(2000)= 2.235106, Δ_N= 0.0216, rel=0.9550%
 n= 50: λ_n(5000)= 42.964009, λ_n(2000)= 42.425262, Δ_N= 0.5387, rel=1.2539%
 n=100: λ_n(5000)= 116.335441, λ_n(2000)= 114.180570, Δ_N= 2.1549, rel=1.8523%
 n=150: λ_n(5000)= 202.573675, λ_n(2000)= 197.725655, Δ_N= 4.8480, rel=2.3932%
 n=200: λ_n(5000)= 297.582659, λ_n(2000)= 288.965050, Δ_N= 8.6176, rel=2.8959%

ζ precision-convergence diagnostics S(n) at N=2000:
 max S(n) over n=1..200: 0.00e+

In [20]:

# Upload artifacts
from e14c.data_storage import upload
uri1 = upload('li_coefficients_N5000_dps50.npz',
 'Li coefficients λ_n (n=1..200) computed from N=5000, dps=50 zero lists for ζ, ζ_δ, L_DH (critical-line zeros) and L_DH augmented with 4 known off-line zeros. Also includes ζ N=2000 truncation comparison.',
 project_tag='-v6-front-ii')
print(uri1)
uri2 = upload('front_ii_li_coefficients.png',
 'Front II final figure: panel A shows λ_n vs n for ζ, ζ_δ, L_DH (critical-line) and L_DH+4 off-line. Panel B shows ζ truncation difference Δ_N(n).',
 project_tag='-v6-front-ii')
print(uri2)


Uploaded li_coefficients_N5000_dps50.npz to data storage as entry data_entry:li-coefficients-n5000-dps50-npz-b6t9 under the name 'li_coefficients_N5000_dps50.npz'.
data_entry:li-coefficients-n5000-dps50-npz-b6t9


Uploaded front_ii_li_coefficients.png to data storage as entry data_entry:front-ii-li-coefficients-png-3nvl under the name 'front_ii_li_coefficients.png'.
data_entry:front-ii-li-coefficients-png-3nvl


In [21]:
from e14c.filesystem.edit_file import edit_file
content = '''## Context
- 2026-05-30 Front II analysis in /workspace/7e5300a5-9b47-4219-b422-9741df37baef.
- Available zero lists confirmed on disk: zeta_zeros_N5000_dps50.npy, zeta_delta_zeros_N5000_dps50.npy, ldh_zeros_N5000_dps50.npy.
- Missing zero lists: L(chi_4 mod 5) N=5000 and L(Delta,s) N=5000. Full 5-way comparative Front II analysis could not be completed because these required inputs were absent.

## Notes
- Provenance: notebook cells #0-19 in current session.
- Computed Li coefficients using lambda_n = sum_rho [1 - (1 - 1/rho)^n], with conjugate-pair reduction 2*Re[1 - (1 - 1/rho)^n], for n<=200.
- Results from available data only:
 - zeta (N=5000): all lambda_n > 0 for n<=200; min = 0.0228688736 at n=1.
 - zeta_delta (N=5000): all lambda_n > 0 for n<=200; min = 0.0228708252 at n=1.
 - L_DH critical-line list (N=5000): all lambda_n > 0 for n<=200; min = 0.0963077258 at n=1.
- Important limitation: ldh_zeros_N5000_dps50.npy contains only critical-line zeros. Li coefficients require summing over all nontrivial zeros. Therefore the negative-Li-coefficient hypothesis for L_DH cannot be decisively tested from this file alone.
- Added the 4 canonical off-line L_DH zeros from -PROMPT-v6-DATASET.md as an exploratory augmentation; still no negative lambda_n for n<=200, but this remains incomplete because the full off-line zero set is not available.
- zeta convergence diagnostic: compared N=2000 vs N=5000; relative truncation difference grew from 0.94% at n=1 to 2.90% at n=200.
- dps diagnostic at N=2000: dps=50 vs dps=80 differences were numerically 0 at float export precision for n<=200.
- Constructed a zero-spectrum Gram-matrix surrogate for Q_N(phi) at N=2000 with Gaussian bump and Fejer kernels; lambda_min = 4.516188 (bump) and -8.75e-14 (Fejer, numerical zero). Caveat: this Gram matrix is PSD by construction and is not the full arithmetic-side Weil quadratic form.
- Artifacts saved: li_coefficients_N5000_dps50.npz and front_ii_li_coefficients.png.
'''
from e14c.filesystem.write_file import write_file
write_file('./.prompts/MEMORY.md', content)
print('memory updated')

ModuleNotFoundError: No module named 'e14c.filesystem.edit_file'; 'e14c.filesystem' is not a package

In [22]:

from e14c.filesystem import write_file
content = '''## Context
- Front II analysis in /workspace/7e5300a5-9b47-4219-b422-9741df37baef.
- Available zero lists: zeta_zeros_N5000_dps50.npy, zeta_delta_zeros_N5000_dps50.npy, ldh_zeros_N5000_dps50.npy.
- Missing: L(chi_4 mod 5) and L(Delta,s) zero lists at N=5000.

## Notes
- Computed Li lambda_n = sum_rho [1 - (1 - 1/rho)^n] using conjugate-pair reduction.
- zeta (N=5000): all lambda_n > 0 for n<=200; min=0.02287 at n=1.
- zeta_delta (N=5000): all lambda_n > 0 for n<=200; min=0.02287 at n=1.
- L_DH critical-line (N=5000): all lambda_n > 0 for n<=200; min=0.09631 at n=1.
- L_DH + 4 known off-line zeros: still all positive.
- Caveat: L_DH file contains only critical-line zeros; full off-line set required for definitive negativity test.
- zeta truncation diagnostic (5k vs 2k): relative diff grows 0.94% (n=1) to 2.90% (n=200).
- dps=50 vs dps=80 at N=2000 zeta: difference ~0 (saturated).
- Weil-form Gram matrix at N=2000 (M=50 test functions): bump min eig=4.52; Fejer min eig ~ -1e-13 (numerical zero). Caveat: Gram matrix is trivially PSD.
- Artifacts: li_coefficients_N5000_dps50.npz, front_ii_li_coefficients.png.
'''
write_file('./.prompts/MEMORY.md', content)
print('ok')


ok
